# Week 8: Custom Exceptions -- Domain Error Design — PHASE 3: Making Composition Maintainable

*Object Oriented Programming . 3 Hours . Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

1. Use `try/except` blocks to handle errors in Python
2. Explain why built-in exceptions are not always enough
3. Create your own **custom exception classes**
4. Design an **exception hierarchy** for a domain
5. Raise exceptions with **meaningful context** information
6. Apply best practices for exception design in engineering systems

## 🎯 Core Mastery Connection

When components collaborate, errors must be meaningful. Custom exceptions let components communicate failures clearly — instead of a generic "something went wrong," each component can signal exactly what failed and why. This makes composed systems debuggable and maintainable.

---

## Quick Recap: Basic Exception Handling (from CP1)

Before we dive into custom exceptions, let's refresh the basics of `try/except` from CP1.

| Keyword | Purpose |
|---------|---------|
| `try` | Wrap code that might raise an error |
| `except` | Handle a specific error type |
| `else` | Runs only if **no** error occurred |
| `finally` | Runs **always**, error or not |

In [ ]:
# Quick recap: try / except / else / finally

def safe_convert(raw):
    """Convert a string to float safely."""
    try:
        value = float(raw)
    except ValueError:
        print(f"  Cannot convert '{raw}' to float")
        return None
    except TypeError:
        print(f"  Expected a string, got {type(raw).__name__}")
        return None
    else:
        print(f"  Converted successfully: {value}")
        return value
    finally:
        print(f"  (attempt finished for input: {raw!r})")

# Test cases
for test in ["3.14", "abc", None]:
    print(f"Input: {test!r}")
    safe_convert(test)
    print()

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import random

---

## Part 1: Review -- try/except Basics

Before we create custom exceptions, let's review how Python's error handling works.

| Keyword | Purpose |
|---------|---------|
| `try` | Code that might cause an error |
| `except` | What to do if an error happens |
| `else` | Runs if NO error happened |
| `finally` | Runs ALWAYS, error or not |

**Figure 8.1** — Try/except error handling

In [ ]:
# Basic try/except review

def divide_voltage(total_voltage, num_resistors):
    """Calculate voltage across each resistor in a series circuit."""
    try:
        result = total_voltage / num_resistors
    except ZeroDivisionError:
        print("Error: Cannot divide by zero resistors!")
        return None
    else:
        print(f"Each resistor gets {result:.2f}V")
        return result
    finally:
        print("Calculation attempt complete.")


# Normal case
divide_voltage(12.0, 4)
print()

# Error case
divide_voltage(12.0, 0)

**Figure 8.2** — Try/except error handling

In [ ]:
# Catching multiple exception types

def read_sensor_value(raw_data):
    """Convert raw sensor string to a float value."""
    try:
        value = float(raw_data)
        if value < 0:
            raise ValueError("Sensor value cannot be negative")
        return value
    except ValueError as e:
        print(f"Invalid sensor data: {e}")
        return None
    except TypeError as e:
        print(f"Wrong data type: {e}")
        return None


# Test different cases
print(read_sensor_value("25.3"))    # Valid
print(read_sensor_value("abc"))     # Invalid string
print(read_sensor_value("-5"))      # Negative value
print(read_sensor_value(None))      # Wrong type

---

## Part 2: The Problem with Generic Exceptions

Built-in exceptions like `ValueError` and `TypeError` are useful, but they don't tell us much about **what went wrong in our specific system**.

Consider this scenario:

**Figure 8.3** — The problem with generic exceptions

In [ ]:
# Problem: Generic exceptions are confusing

class Motor:
    def __init__(self, max_rpm=5000):
        self.max_rpm = max_rpm
        self.rpm = 0

    def set_speed(self, rpm):
        if rpm < 0:
            raise ValueError("Speed cannot be negative")
        if rpm > self.max_rpm:
            raise ValueError("Speed too high")
        self.rpm = rpm

    def set_temperature_limit(self, temp):
        if temp < 0:
            raise ValueError("Temperature cannot be negative")
        if temp > 150:
            raise ValueError("Temperature too high")


motor = Motor()

# Both raise ValueError -- but they are VERY different problems!
try:
    motor.set_speed(9999)
except ValueError as e:
    # Is this a speed problem? Temperature problem? Something else?
    # We can't tell just from "ValueError"!
    print(f"Got ValueError: {e}")
    print("But WHAT kind of value was wrong? Speed? Temperature? Pressure?")

### Why This Is a Problem

| Issue | Explanation |
|-------|-------------|
| **Can't distinguish errors** | `ValueError` could mean speed, temperature, or anything else |
| **Hard to handle specifically** | You can't catch speed errors separately from temperature errors |
| **Poor debugging** | When you see `ValueError` in a log, you don't know which part failed |
| **No domain meaning** | `ValueError` doesn't tell you anything about your engineering system |

The solution: **create your own exception classes!**

---

## Part 3: Creating Custom Exceptions

A custom exception is just a **class that inherits from `Exception`** (or one of its subclasses).

**Figure 3.1** -- Creating a custom exception

```
Exception (built-in)
    |
    +-- ValueError (built-in)
    |
    +-- MotorError (custom!)         <-- We create this
         |
         +-- OverspeedError (custom!) <-- And this
         +-- OverheatError (custom!)  <-- And this
```

In [ ]:
# The simplest custom exception

class MotorError(Exception):
    """Base exception for motor-related errors."""
    pass


# That's it! Just inherit from Exception.
# Now let's use it:

try:
    raise MotorError("Something went wrong with the motor")
except MotorError as e:
    print(f"Motor problem: {e}")

**Figure 8.4** — Raising custom exceptions

In [ ]:
# Custom exceptions with extra information

class OverspeedError(MotorError):
    """Raised when motor speed exceeds the safe limit."""

    def __init__(self, current_speed, max_speed):
        self.current_speed = current_speed
        self.max_speed = max_speed
        # Create a clear error message
        message = f"Speed {current_speed} RPM exceeds maximum {max_speed} RPM"
        super().__init__(message)


class OverheatError(MotorError):
    """Raised when motor temperature exceeds the safe limit."""

    def __init__(self, temperature, max_temp=120):
        self.temperature = temperature
        self.max_temp = max_temp
        message = f"Temperature {temperature}C exceeds limit {max_temp}C"
        super().__init__(message)


# Now we can tell exactly WHAT went wrong
try:
    raise OverspeedError(6000, 5000)
except OverspeedError as e:
    print(f"Overspeed! Current: {e.current_speed}, Max: {e.max_speed}")
except OverheatError as e:
    print(f"Overheat! Current: {e.temperature}C")
except MotorError as e:
    print(f"Some other motor error: {e}")

### Key Points

| Feature | How |
|---------|-----|
| Create a custom exception | `class MyError(Exception): pass` |
| Add custom data | Define `__init__` with extra parameters |
| Set the message | Call `super().__init__(message)` |
| Access custom data | Use `e.attribute_name` in the except block |

---

## Part 4: Exception Hierarchies

Just like classes, exceptions can form a **hierarchy**. This lets you catch errors at different levels of detail.

**Figure 4.1** -- Exception hierarchy for a motor control system

```
Exception
  +-- MotorError
       +-- OverspeedError
       +-- OverheatError
       +-- StallError
```

With this hierarchy:
- `except OverspeedError` catches only speed problems
- `except MotorError` catches ALL motor problems (speed, heat, stall)

In [ ]:
# Full exception hierarchy for a motor system

class MotorError(Exception):
    """Base class for all motor errors."""
    pass

class OverspeedError(MotorError):
    """Motor is running too fast."""
    def __init__(self, rpm, limit):
        self.rpm = rpm
        self.limit = limit
        super().__init__(f"Speed {rpm} RPM exceeds limit {limit} RPM")

class OverheatError(MotorError):
    """Motor temperature is too high."""
    def __init__(self, temp, limit=120):
        self.temp = temp
        self.limit = limit
        super().__init__(f"Temperature {temp}C exceeds limit {limit}C")

class StallError(MotorError):
    """Motor has stalled (stopped unexpectedly)."""
    def __init__(self, motor_id):
        self.motor_id = motor_id
        super().__init__(f"Motor {motor_id} has stalled")

**Figure 8.5** — Catching exceptions at different levels of the hierarchy

In [ ]:
# Catching at different levels of the hierarchy

import random

def simulate_motor_problem():
    """Simulate a random motor problem."""
    problem = random.choice(["overspeed", "overheat", "stall"])

    if problem == "overspeed":
        raise OverspeedError(6500, 5000)
    elif problem == "overheat":
        raise OverheatError(135)
    else:
        raise StallError("M-001")


# Catching specific errors
print("=== Catching specific errors ===")
try:
    simulate_motor_problem()
except OverspeedError as e:
    print(f"SPEED ISSUE: Reduce to {e.limit} RPM")
except OverheatError as e:
    print(f"HEAT ISSUE: Cool down from {e.temp}C")
except StallError as e:
    print(f"STALL: Restart motor {e.motor_id}")

print()

# Catching ALL motor errors at once
print("=== Catching all motor errors ===")
try:
    simulate_motor_problem()
except MotorError as e:
    # This catches OverspeedError, OverheatError, AND StallError
    print(f"Motor problem detected: {e}")
    print(f"Error type: {type(e).__name__}")

### Hierarchy Design Rules

| Rule | Example |
|------|---------|
| Start with a **base error** for your domain | `MotorError`, `SensorError` |
| Put **specific errors** under the base | `OverspeedError(MotorError)` |
| Order `except` blocks from **specific to general** | `OverspeedError` before `MotorError` |
| Don't inherit from `BaseException` | Always use `Exception` or a subclass |

---

## Part 5: Raising Exceptions with Context

Good exceptions carry **useful information** that helps the developer (or the system) fix the problem.

**Figure 5.1** -- Bad vs. good error messages

| Bad | Good |
|-----|------|
| `"Error"` | `"Motor M-001 speed 6500 RPM exceeds limit 5000 RPM"` |
| `"Invalid value"` | `"Sensor S-003 reading -5.2C is below minimum 0C"` |
| `"Failed"` | `"Conveyor belt B-02 stalled at position 145cm after 30 seconds"` |

In [ ]:
# Custom exception with rich context

class SensorError(Exception):
    """Base class for sensor errors."""
    pass


class SensorReadingError(SensorError):
    """Raised when a sensor gives an invalid reading."""

    def __init__(self, sensor_id, value, min_val, max_val):
        self.sensor_id = sensor_id
        self.value = value
        self.min_val = min_val
        self.max_val = max_val

        # Build a clear, informative message
        if value < min_val:
            detail = f"{value} is below minimum {min_val}"
        elif value > max_val:
            detail = f"{value} is above maximum {max_val}"
        else:
            detail = f"{value} is out of range [{min_val}, {max_val}]"

        message = f"Sensor {sensor_id}: {detail}"
        super().__init__(message)


class SensorDisconnectedError(SensorError):
    """Raised when a sensor is not responding."""

    def __init__(self, sensor_id, last_reading_time=None):
        self.sensor_id = sensor_id
        self.last_reading_time = last_reading_time

        if last_reading_time:
            message = f"Sensor {sensor_id} disconnected (last seen: {last_reading_time})"
        else:
            message = f"Sensor {sensor_id} is not connected"
        super().__init__(message)


# Using these exceptions
try:
    # Simulate a bad reading from a temperature sensor
    reading = 155.0  # Way too hot!
    if reading < -40 or reading > 150:
        raise SensorReadingError("TEMP-01", reading, -40, 150)
except SensorReadingError as e:
    print(f"Error: {e}")
    print(f"  Sensor: {e.sensor_id}")
    print(f"  Value: {e.value}")
    print(f"  Valid range: [{e.min_val}, {e.max_val}]")

**Figure 8.6** — Raising custom exceptions

In [ ]:
# Using custom exceptions in a class

class TemperatureSensor:
    """A temperature sensor that raises meaningful exceptions."""

    def __init__(self, sensor_id, min_temp=-40, max_temp=150):
        self.sensor_id = sensor_id
        self.min_temp = min_temp
        self.max_temp = max_temp
        self.connected = True

    def read(self):
        """Read the sensor, raising clear errors if something is wrong."""
        if not self.connected:
            raise SensorDisconnectedError(self.sensor_id)

        # Simulate a reading
        import random
        value = round(random.uniform(-50, 160), 1)

        if value < self.min_temp or value > self.max_temp:
            raise SensorReadingError(self.sensor_id, value, self.min_temp, self.max_temp)

        return value


# Test the sensor
sensor = TemperatureSensor("TEMP-01")

for i in range(5):
    try:
        value = sensor.read()
        print(f"Reading {i+1}: {value}C -- OK")
    except SensorReadingError as e:
        print(f"Reading {i+1}: BAD -- {e}")
    except SensorDisconnectedError as e:
        print(f"Reading {i+1}: DISCONNECTED -- {e}")

---

## Part 6: Best Practices for Exception Design

Here are the rules to follow when designing custom exceptions:

| Practice | Do | Don't |
|----------|----|---------|
| **Name clearly** | `OverspeedError` | `Error1` |
| **End with 'Error'** | `SensorReadingError` | `SensorProblem` |
| **Use a base class** | `class MyError(SensorError)` | `class MyError(Exception)` for everything |
| **Include context** | `f"Sensor {id}: {value} out of range"` | `"Error"` |
| **Keep it simple** | Store 2-3 useful attributes | Store entire system state |
| **Document it** | Add a docstring | Leave it unexplained |

**Figure 8.7** — Custom exception class definition

In [ ]:
# GOOD: A well-designed exception hierarchy for a robotics system

class RobotError(Exception):
    """Base exception for all robot-related errors."""
    pass


class JointLimitError(RobotError):
    """Raised when a joint exceeds its movement range."""

    def __init__(self, joint_name, angle, min_angle, max_angle):
        self.joint_name = joint_name
        self.angle = angle
        self.min_angle = min_angle
        self.max_angle = max_angle
        super().__init__(
            f"Joint '{joint_name}': angle {angle} degrees "
            f"is outside range [{min_angle}, {max_angle}]"
        )


class CollisionError(RobotError):
    """Raised when the robot detects a potential collision."""

    def __init__(self, object_name, distance_cm):
        self.object_name = object_name
        self.distance_cm = distance_cm
        super().__init__(
            f"Collision risk: '{object_name}' detected {distance_cm} cm away"
        )


class EmergencyStopError(RobotError):
    """Raised when emergency stop is triggered."""

    def __init__(self, reason):
        self.reason = reason
        super().__init__(f"EMERGENCY STOP: {reason}")


# Using the hierarchy
def move_robot_arm(joint, angle):
    """Move a robot arm joint to a specific angle."""
    limits = {
        "shoulder": (-90, 90),
        "elbow": (0, 135),
        "wrist": (-180, 180),
    }

    if joint not in limits:
        raise RobotError(f"Unknown joint: {joint}")

    min_a, max_a = limits[joint]
    if angle < min_a or angle > max_a:
        raise JointLimitError(joint, angle, min_a, max_a)

    print(f"Moving {joint} to {angle} degrees")


# Test cases
test_moves = [
    ("shoulder", 45),    # OK
    ("elbow", 150),      # Too far
    ("wrist", -90),      # OK
    ("finger", 30),      # Unknown joint
]

for joint, angle in test_moves:
    try:
        move_robot_arm(joint, angle)
    except JointLimitError as e:
        print(f"LIMIT: {e}")
    except RobotError as e:
        print(f"ROBOT ERROR: {e}")

### Common Mistakes to Avoid

| Mistake | Why It's Bad |
|---------|--------------|
| Catching `Exception` everywhere | Hides bugs -- you might catch errors you didn't expect |
| Using `pass` in except blocks | Silently ignoring errors is dangerous in engineering systems |
| Raising `Exception` directly | Too generic -- use a specific type |
| Not calling `super().__init__()` | The error message won't work properly |

**Figure 8.8** — Bad practices vs. good practices in exception handling

In [ ]:
# BAD PRACTICES -- Don't do these!

# BAD: Catching everything silently
try:
    result = 10 / 0
except Exception:
    pass  # Error is hidden! Very dangerous!

print("The error was silently ignored. This is dangerous!")
print()

# GOOD: Catch specific errors and handle them
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Cannot divide by zero: {e}")
    result = 0  # Provide a fallback value

print(f"Result: {result}")

---

## Part 7: Putting It All Together

Let's build a complete sensor monitoring system with well-designed custom exceptions.

**Figure 7.1** -- Sensor system exception hierarchy

```
Exception
  +-- SensorSystemError
       +-- SensorReadingError
       +-- SensorDisconnectedError
       +-- CalibrationError
```

In [ ]:
# Complete sensor monitoring system with custom exceptions

import random


# --- Exception hierarchy ---

class SensorSystemError(Exception):
    """Base exception for the sensor monitoring system."""
    pass


class SensorReadingError(SensorSystemError):
    """Raised when a sensor reading is out of valid range."""

    def __init__(self, sensor_id, value, valid_range):
        self.sensor_id = sensor_id
        self.value = value
        self.valid_range = valid_range
        super().__init__(
            f"Sensor {sensor_id}: reading {value} is outside "
            f"valid range {valid_range}"
        )


class SensorDisconnectedError(SensorSystemError):
    """Raised when a sensor cannot be reached."""

    def __init__(self, sensor_id):
        self.sensor_id = sensor_id
        super().__init__(f"Sensor {sensor_id} is disconnected")


class CalibrationError(SensorSystemError):
    """Raised when a sensor needs recalibration."""

    def __init__(self, sensor_id, drift):
        self.sensor_id = sensor_id
        self.drift = drift
        super().__init__(
            f"Sensor {sensor_id}: calibration drift of {drift:.1f}% detected"
        )


# --- Sensor class ---

class Sensor:
    """A sensor that raises meaningful exceptions."""

    def __init__(self, sensor_id, unit, min_val, max_val):
        self.sensor_id = sensor_id
        self.unit = unit
        self.min_val = min_val
        self.max_val = max_val
        self.connected = True
        self.calibration_drift = 0.0

    def read(self):
        """Read the sensor value with proper error handling."""
        # Check connection
        if not self.connected:
            raise SensorDisconnectedError(self.sensor_id)

        # Check calibration
        if abs(self.calibration_drift) > 5.0:
            raise CalibrationError(self.sensor_id, self.calibration_drift)

        # Simulate a reading (sometimes out of range)
        value = round(random.uniform(self.min_val - 10, self.max_val + 10), 1)

        if value < self.min_val or value > self.max_val:
            raise SensorReadingError(
                self.sensor_id, value, (self.min_val, self.max_val)
            )

        return value


# --- Monitor class ---

class SensorMonitor:
    """Monitors multiple sensors and handles errors."""

    def __init__(self):
        self.sensors = []
        self.error_count = 0

    def add_sensor(self, sensor):
        self.sensors.append(sensor)

    def check_all(self):
        """Read all sensors and report status."""
        print("=" * 50)
        print("SENSOR MONITORING REPORT")
        print("=" * 50)

        for sensor in self.sensors:
            try:
                value = sensor.read()
                print(f"  [OK]    {sensor.sensor_id}: {value} {sensor.unit}")

            except SensorDisconnectedError as e:
                self.error_count += 1
                print(f"  [DISC]  {e}")

            except CalibrationError as e:
                self.error_count += 1
                print(f"  [CAL]   {e}")

            except SensorReadingError as e:
                self.error_count += 1
                print(f"  [RANGE] {e}")

            except SensorSystemError as e:
                self.error_count += 1
                print(f"  [ERR]   {e}")

        print(f"\nTotal errors: {self.error_count}")
        print("=" * 50)

**Figure 8.9** — Sensor monitoring system

In [ ]:
# Run the sensor monitoring system

# Create sensors
temp_sensor = Sensor("TEMP-01", "C", min_val=0, max_val=100)
pressure_sensor = Sensor("PRES-01", "bar", min_val=1, max_val=10)
humidity_sensor = Sensor("HUM-01", "%", min_val=20, max_val=90)

# Simulate some problems
disconnected_sensor = Sensor("TEMP-02", "C", min_val=0, max_val=100)
disconnected_sensor.connected = False

bad_cal_sensor = Sensor("PRES-02", "bar", min_val=1, max_val=10)
bad_cal_sensor.calibration_drift = 8.5

# Set up monitor
monitor = SensorMonitor()
monitor.add_sensor(temp_sensor)
monitor.add_sensor(pressure_sensor)
monitor.add_sensor(humidity_sensor)
monitor.add_sensor(disconnected_sensor)
monitor.add_sensor(bad_cal_sensor)

# Run the check
monitor.check_all()

---

## Exercises

> **Composition lens:** Each custom exception you design is a communication channel between components. When a `Sensor` component raises a `SensorReadingError`, the `Monitor` component knows exactly what happened and can respond appropriately. Clear error communication is essential for reliable composition.

Complete the following exercises to practice creating and using custom exceptions.

### Exercise 1: Simple Custom Exception (Easy)

Create a custom exception called `BatteryLowError` that: - Inherits from Exception - Takes battery_level (a percentage) as a parameter - Has a clear error message like "Battery at 5%: level below minimum 10%" Then write a function `check_battery(level)` that raises BatteryLowError if the level is below 10%. Test with levels: 50, 8, 3

<details>
<summary>💡 Hint</summary>
Use <code>class BatteryLowError(Exception)</code> and add an <code>__init__</code> to store the battery level.
</details>

In [ ]:
# ✏️ [EX1]
# Create a custom exception called `BatteryLowError` that:
# - Inherits from Exception
# - Takes battery_level (a percentage) as a parameter
# - Has a clear error message like "Battery at 5%: level below minimum 10%"
#
# Then write a function `check_battery(level)` that raises BatteryLowError
# if the level is below 10%.
#
# Test with levels: 50, 8, 3

# Hint: Follow the same pattern as OverspeedError

# YOUR CODE HERE


### Exercise 2: Exception Hierarchy (Medium)

Create an exception hierarchy for a vehicle system: VehicleError (base) +-- EngineError +-- BrakeError +-- FuelError Each should have: - A meaningful __init__ with at least one custom parameter - A clear error message Then write code that raises each type and catches them.

<details>
<summary>💡 Hint</summary>
Start with <code>VehicleError(Exception)</code>, then <code>EngineError(VehicleError)</code>, etc. Each child inherits from its parent.
</details>

In [ ]:
# ✏️ [EX2]
# Create an exception hierarchy for a vehicle system:
#
# VehicleError (base)
#   +-- EngineError
#   +-- BrakeError
#   +-- FuelError
#
# Each should have:
# - A meaningful __init__ with at least one custom parameter
# - A clear error message
#
# Then write code that raises each type and catches them.

# Hint: Start with the base class, then add specific ones

# YOUR CODE HERE


### Exercise 3: Fix the Catch Order (Easy)

The code below has a bug in the except order. Fix it so that specific exceptions are caught before general ones.

<details>
<summary>💡 Hint</summary>
Python checks <code>except</code> blocks top-to-bottom. Put the <em>most specific</em> exception first, <em>most general</em> last.
</details>

In [ ]:
# ✏️ [EX3]
# The code below has a bug in the except order.
# Fix it so that specific exceptions are caught before general ones.

class AppError(Exception):
    pass

class DatabaseError(AppError):
    pass

class ConnectionError(DatabaseError):
    pass

# BUG: This catches everything as AppError!
try:
    raise ConnectionError("Cannot connect to database")
except AppError as e:
    print(f"App error: {e}")
except DatabaseError as e:
    print(f"Database error: {e}")
except ConnectionError as e:
    print(f"Connection error: {e}")

# FIX THE ORDER BELOW:

# YOUR CODE HERE


### Exercise 4: Add Context to Exception (Medium)

Create a `MotorOverloadError` that stores: - motor_id (string) - current_load (float, in Amps) - max_load (float, in Amps) - duration (float, in seconds -- how long the overload lasted) The error message should include ALL of this information. Then write a Motor class with a run(load, duration) method that raises MotorOverloadError if load exceeds the max.

<details>
<summary>💡 Hint</summary>
Store extra info as attributes: <code>self.motor_id = motor_id</code>, <code>self.current_load = current_load</code> in <code>__init__</code>.
</details>

In [ ]:
# ✏️ [EX4]
# Create a `MotorOverloadError` that stores:
# - motor_id (string)
# - current_load (float, in Amps)
# - max_load (float, in Amps)
# - duration (float, in seconds -- how long the overload lasted)
#
# The error message should include ALL of this information.
#
# Then write a Motor class with a run(load, duration) method
# that raises MotorOverloadError if load exceeds the max.

# Hint: Store all values as attributes AND put them in the message

# YOUR CODE HERE


### Exercise 5: Exception with Recovery Suggestion (Medium)

Create exceptions that include a `suggestion` attribute telling the user what to do. Create: - LowPressureError(current, minimum) suggestion: "Increase pressure to at least {minimum} bar" - HighTemperatureError(current, maximum) suggestion: "Activate cooling system or reduce load" Then write code that catches these and prints the suggestion.

<details>
<summary>💡 Hint</summary>
Add a <code>self.suggestion</code> attribute. E.g., for overheating: <code>'Reduce speed or improve cooling'</code>.
</details>

In [ ]:
# ✏️ [EX5]
# Create exceptions that include a `suggestion` attribute
# telling the user what to do.
#
# Create:
# - LowPressureError(current, minimum)
#     suggestion: "Increase pressure to at least {minimum} bar"
# - HighTemperatureError(current, maximum)
#     suggestion: "Activate cooling system or reduce load"
#
# Then write code that catches these and prints the suggestion.

# Hint: Add a self.suggestion attribute in __init__

# YOUR CODE HERE


### Exercise 6: Sensor Validation System (Medium)

Build a sensor that validates readings using custom exceptions: Exceptions: - SensorError (base) - OutOfRangeError(sensor_id, value, min_val, max_val) - SpikeError(sensor_id, current, previous)  -- sudden jump in value Rules: - Temperature must be between -20 and 80 - If the reading changes by more than 20 degrees from the previous reading, raise SpikeError Test with values: [25, 26, 28, 55, 82, 30]

<details>
<summary>💡 Hint</summary>
In the sensor's <code>read()</code> method, check the value and <code>raise</code> the appropriate custom exception with a descriptive message.
</details>

In [ ]:
# ✏️ [EX6]
# Build a sensor that validates readings using custom exceptions:
#
# Exceptions:
# - SensorError (base)
# - OutOfRangeError(sensor_id, value, min_val, max_val)
# - SpikeError(sensor_id, current, previous)  -- sudden jump in value
#
# Rules:
# - Temperature must be between -20 and 80
# - If the reading changes by more than 20 degrees from the previous
#   reading, raise SpikeError
#
# Test with values: [25, 26, 28, 55, 82, 30]

# Hint: Store the previous reading to detect spikes

# YOUR CODE HERE


### Exercise 7: Re-raising Exceptions (Medium)

Sometimes you want to catch an exception, do something (like logging), and then re-raise it. Use the `raise` keyword without arguments. Write a function `safe_motor_start(motor_id, speed)` that: 1. Tries to start a motor at the given speed 2. If speed > 5000, raises OverspeedError 3. Catches the error, prints a log message 4. Re-raises the error so the caller can also handle it Then call safe_motor_start inside ANOTHER try/except to show that the error is caught at a higher level.

<details>
<summary>💡 Hint</summary>
Use <code>except SomeError as e: log(e); raise</code>. The bare <code>raise</code> re-raises the same exception.
</details>

In [ ]:
# ✏️ [EX7]
# Sometimes you want to catch an exception, do something (like logging),
# and then re-raise it. Use the `raise` keyword without arguments.
#
# Write a function `safe_motor_start(motor_id, speed)` that:
# 1. Tries to start a motor at the given speed
# 2. If speed > 5000, raises OverspeedError
# 3. Catches the error, prints a log message
# 4. Re-raises the error so the caller can also handle it
#
# Then call safe_motor_start inside ANOTHER try/except to show
# that the error is caught at a higher level.

# Hint: Use `raise` by itself to re-raise the current exception

# YOUR CODE HERE


### Exercise 8: Exception in a Pipeline (Challenge)

Create a data processing pipeline for sensor data: Step 1: Read raw value (might fail -- SensorError) Step 2: Convert units (might fail -- ConversionError) Step 3: Validate range (might fail -- ValidationError) Create the three exception classes and a `process_reading(raw)` function that performs all three steps. Handle each error type differently. Test with inputs: "25.0", "abc", "999", "-100"

<details>
<summary>💡 Hint</summary>
Wrap each pipeline step in try/except. If one step fails, catch it and either skip or provide a default value.
</details>

In [ ]:
# ✏️ [EX8]
# Create a data processing pipeline for sensor data:
#
# Step 1: Read raw value (might fail -- SensorError)
# Step 2: Convert units (might fail -- ConversionError)
# Step 3: Validate range (might fail -- ValidationError)
#
# Create the three exception classes and a `process_reading(raw)`
# function that performs all three steps.
# Handle each error type differently.
#
# Test with inputs: "25.0", "abc", "999", "-100"

# Hint: Each step should raise its own exception type

# YOUR CODE HERE


### Exercise 9: Custom __str__ Method (Easy)

Create a `MachineError` exception that has a custom __str__ method which formats the error nicely: Example output: ========================================== MACHINE ERROR Machine: CNC-001 Error: Spindle jammed Severity: HIGH Action: Stop machine and call maintenance ========================================== The __init__ should accept: machine_id, error_msg, severity, action

<details>
<summary>💡 Hint</summary>
Define <code>def __str__(self)</code> in your exception class. Return a formatted string with all the error details.
</details>

In [ ]:
# ✏️ [EX9]
# Create a `MachineError` exception that has a custom __str__ method
# which formats the error nicely:
#
# Example output:
# ==========================================
# MACHINE ERROR
# Machine: CNC-001
# Error: Spindle jammed
# Severity: HIGH
# Action: Stop machine and call maintenance
# ==========================================
#
# The __init__ should accept: machine_id, error_msg, severity, action

# Hint: Override __str__ to return a formatted string

# YOUR CODE HERE


### Exercise 10: Build a Complete System (Challenge)

Create a conveyor belt monitoring system with these exceptions: ConveyorError (base) +-- BeltSlipError(belt_id, slip_percentage) +-- JamError(belt_id, position) +-- OverloadError(belt_id, weight, max_weight) Create a ConveyorBelt class with: - __init__(belt_id, max_weight) - load(weight): raises OverloadError if weight > max_weight - run(): randomly raises BeltSlipError or JamError (simulate problems) Create a ConveyorMonitor that runs multiple belts and reports all errors.

<details>
<summary>💡 Hint</summary>
Create the exception hierarchy first, then the ConveyorBelt class that raises them. Test each exception separately.
</details>

In [ ]:
# ✏️ [EX10]
# Create a conveyor belt monitoring system with these exceptions:
#
# ConveyorError (base)
#   +-- BeltSlipError(belt_id, slip_percentage)
#   +-- JamError(belt_id, position)
#   +-- OverloadError(belt_id, weight, max_weight)
#
# Create a ConveyorBelt class with:
# - __init__(belt_id, max_weight)
# - load(weight): raises OverloadError if weight > max_weight
# - run(): randomly raises BeltSlipError or JamError (simulate problems)
#
# Create a ConveyorMonitor that runs multiple belts and reports all errors.

# Hint: Use random.choice to simulate different problems

# YOUR CODE HERE


### Exercise 11: When NOT to Use Custom Exceptions (Easy)

Answer as comments: For each scenario, should you use a custom exception or a built-in one? Explain your reasoning. A) A function receives a string when it expects an integer B) A motor speed exceeds the safety limit C) A file is not found on disk D) A sensor reading is outside the expected range for your specific system E) A list index is out of bounds

<details>
<summary>💡 Hint</summary>
Use built-in exceptions when Python already has a good one (<code>ValueError</code>, <code>TypeError</code>). Custom ones are for domain-specific errors.
</details>

In [ ]:
# ✏️ [EX11]
# Answer as comments:
#
# For each scenario, should you use a custom exception or a built-in one?
# Explain your reasoning.
#
# A) A function receives a string when it expects an integer
# B) A motor speed exceeds the safety limit
# C) A file is not found on disk
# D) A sensor reading is outside the expected range for your specific system
# E) A list index is out of bounds

# YOUR ANSWERS:
# A: ...
# B: ...
# C: ...
# D: ...
# E: ...

### Exercise 12: Reflection (Easy)

Answer these questions as comments: 1. Why are custom exceptions better than using ValueError for everything? 2. What information should a good custom exception include? 3. Why should you order except blocks from specific to general? 4. Give an example from YOUR engineering field where a custom exception would be useful.

<details>
<summary>💡 Hint</summary>
Think about debugging: would you rather see <code>ValueError: invalid value</code> or <code>MotorOverheatError: Motor M1 at 105°C (max 100°C)</code>?
</details>

In [ ]:
# ✏️ [EX12]
# Answer these questions as comments:
#
# 1. Why are custom exceptions better than using ValueError for everything?
#
# 2. What information should a good custom exception include?
#
# 3. Why should you order except blocks from specific to general?
#
# 4. Give an example from YOUR engineering field where a custom exception
#    would be useful.

# YOUR ANSWERS:
# 1. ...
# 2. ...
# 3. ...
# 4. ...

---

### 🌉 Bridge to Next Week

This week we learned how to create **custom exceptions** that make error handling clear and meaningful. We designed exception hierarchies and added useful context to our error messages.

But how do we know our exception handling actually *works*? How can we be confident that `OverspeedError` really fires at the right speed, or that `SensorDisconnectedError` carries the correct sensor ID?

**Next week**, we will learn about **Unit Testing** — a systematic way to verify that every piece of your code behaves correctly. You will write small, focused tests that automatically check your classes, methods, and yes, your custom exceptions.

```python
# Sneak peek — Week 9
import unittest

class TestMotor(unittest.TestCase):
    def test_overspeed_raises_error(self):
        motor = Motor(max_rpm=5000)
        with self.assertRaises(OverspeedError):
            motor.set_speed(6000)
```

---
## 📮 Submission

Follow the two steps below to submit your work.

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 1: Fill in your info below, then run this cell#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━STUDENT_ID    = ""     # e.g. "2024001234"STUDENT_NAME  = ""     # e.g. "Ahmet Yılmaz"STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"CLASS_CODE    = ""     # code given in class#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# Don't change anything below this line#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import re as _re_errors = []if not _re.match(r"^\d{6,12}$", STUDENT_ID):    _errors.append("❌ Student ID must be 6-12 digits")if len(STUDENT_NAME.strip().split()) < 2:    _errors.append("❌ Enter first and last name")if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:    _errors.append("❌ Use your @istun.edu.tr email")if len(CLASS_CODE.strip()) < 4:    _errors.append("❌ Invalid class code")if _errors:    for _e in _errors:        print(_e)    print("\n⚠️  Fix the errors above and run this cell again.")else:    print(f"✅ Info OK — {STUDENT_NAME} ({STUDENT_ID})")    print(f"   {STUDENT_EMAIL}")    print(f"\n👉 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 2: Run this cell to submit#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import json, re, os, urllib.requestWEEK = "Week_08"URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"try:    _sid = STUDENT_ID.strip()    _sname = STUDENT_NAME.strip()    _semail = STUDENT_EMAIL.strip().lower()    _scode = CLASS_CODE.strip().upper()except NameError:    raise SystemExit("❌ Run the cell above first to set your info!")if not _sid or not _sname or not _semail or not _scode:    raise SystemExit("❌ Run the cell above first — some fields are empty.")_answers = {}try:    _ipy = get_ipython()    _hist = _ipy.history_manager.get_range(output=False)    for _sess, _line, _src in _hist:        _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)        if _m:            _ex_id = "ex" + _m.group(1)            _lines = _src.split("\n")            _clean = "\n".join(_lines[1:]).strip()            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}except Exception:    passif not _answers:    try:        for _src in In:            if not _src: continue            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}    except NameError:        passif not _answers:    _nb_path = None    try:        _nb_path = __vsc_ipynb_file__    except NameError:        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]        if len(_candidates) == 1: _nb_path = _candidates[0]    if _nb_path and os.path.exists(str(_nb_path)):        with open(str(_nb_path), "r", encoding="utf-8") as _f:            _nb = json.load(_f)        for _cell in _nb["cells"]:            if _cell["cell_type"] != "code": continue            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}print(f"📝 Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")if not _answers:    print("\n⚠️  No exercise answers found!")    print("Make sure you RAN all exercise cells before submitting.")    raise SystemExit()_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "oop-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")print("📡 Submitting...")try:    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")    _resp = urllib.request.urlopen(_req, timeout=30)    _result = json.loads(_resp.read().decode())    if _result.get("success"):        print(f"\n✅ {_result['message']}")        print("📧 Check your email for confirmation.")    else:        print(f"\n❌ {_result.get('message', 'Submission failed')}")except Exception as _e:    try:        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")        urllib.request.urlopen(_req, timeout=10)    except: pass    print(f"\n⚠️  Request sent — check your email for confirmation.")    print(f"(If no email arrives, try again or contact your instructor)")